In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from gps_ids import figures
import seaborn as sns
from sklearn.metrics import classification_report, accuracy_score, f1_score


In [ ]:
from gps_ids import protocol, training

EXP = protocol.real_heldout("2d")
r = training.run("XGBoost", EXP)
model = r.model
y_pred = r.y_pred
y_test = EXP.y_test
feature_names = EXP.feature_names


In [7]:
# Report the model's performance on the test set
print("\nEvaluating the model on the test set...")
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print(f"Model Accuracy on Test Set: {accuracy * 100:.2f}%")
print(f"F1 Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


Evaluating the model on the test set...
Model Accuracy on Test Set: 13.15%
F1 Score: 0.1290

Classification Report:
              precision    recall  f1-score   support

           0       0.64      0.09      0.15    318260
           1       0.00      0.00      0.00     29166
           2       0.15      0.00      0.00     35386
           3       0.07      1.00      0.13     25612

    accuracy                           0.13    408424
   macro avg       0.21      0.27      0.07    408424
weighted avg       0.51      0.13      0.13    408424



In [ ]:
importances = model.feature_importances_

# Create a DataFrame for feature importances
feat_imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
})

# Sort by importance descending
feat_imp_df = feat_imp_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=feat_imp_df.head(10),
    x='Importance',
    y='Feature',
    palette='mako',
    hue='Feature',
    legend=False
)
plt.title('Top 10 Feature Importances')
plt.tight_layout()
figures.save('feature_importance')
plt.show()

In [10]:
print(feat_imp_df.head(10))

   Feature  Importance
4      TOW    0.223913
2       PD    0.196729
3       RX    0.157455
0      PRN    0.142636
11     TCD    0.113491
1       DO    0.074641
5       CP    0.052822
12     CN0    0.020185
8       PC    0.004042
7       LC    0.003671


In [ ]:
# Seed sweep: the remaining seeds in training.SEEDS, appended to the results CSV.
SWEEP_SEEDS = [s for s in training.SEEDS if s != protocol.SEED]

training.run_seeds("XGBoost", lambda s: protocol.real_heldout("2d", seed=s), seeds=SWEEP_SEEDS)
